# 09 — SEC breakpoint robustness analysis

This notebook quantifies which breakpoints survive perturbation of the
processing pipeline (savgol vs lowess; N=1..10). Breakpoints that
appear repeatedly across both methods and many N values are the most
defensible "geological" interfaces; the rest are likely method-induced.

## Outputs

- ``robustness_clusters.csv`` — per-cluster scores (persistence, agreement)
- ``robustness_bp_assignments.csv`` — per-BP assignments (full traceability)
- ``robustness_summary.csv`` — BIC-optimal N per (well, smoothing)
- ``sensitivity_clusters.csv`` — clusters at δ ∈ {0.3, 0.5, 1.0} m
- Per-well panel figures + δ-sensitivity figures + BIC curves

## Convention reminder

Y-axis is BGL-positive (depth in metres below ground level, 0 at top).

## Scientific reading

- ``persistence`` ∈ [0, 20] = total count of (smoothing, N) combinations that
  detect a BP in this cluster. Max 20 = 10 N × 2 smoothings.
- ``agreement`` ∈ [0, 10] = min(n_savgol, n_lowess). Penalises one-sided
  detections. **This is the primary score for "robustness"**.
- ``wide_flag`` = True if cluster spans > 2 m. Marks transition zones
  (e.g. continuous karst sections where the SEC gradient is irregular)
  rather than discrete breakpoints.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

from karst_analysis.sec.robustness import (
    compute_robustness, compute_robustness_sensitivity,
    plot_robustness_panel, plot_delta_sensitivity, plot_bic_curves,
)
from karst_analysis.convergence import WELLS

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print(f"CWD: {Path.cwd()}")

## 1. Single-well analysis

In [ ]:
res = compute_robustness("LRS70D", campaign="2022_02", delta_m=0.5)

print(f"BP records pooled: {len(res.bp_records)}")
print(f"Clusters found at δ=0.5 m: {len(res.clusters)}")
print(f"\nN max observed per smoothing: {res.n_max_smoothing}")
print()
print("Top 6 clusters by agreement:")
res.clusters.head(6)

In [ ]:
# BIC summary: where does the BIC sweep find its minimum N?
res.bic_summary

## 2. Per-well diagnostic panel

In [ ]:
fig = plot_robustness_panel(
    res, well_id="LRS70D",
    output_path="results/sec_robustness/2022_02/figures/robustness_LRS70D.png",
)
plt.close(fig)

from IPython.display import Image
Image(filename="results/sec_robustness/2022_02/figures/robustness_LRS70D.png")

## 3. δ-sensitivity scan

Run the same analysis at δ ∈ {0.3, 0.5, 1.0} m. The qualitative
result should be similar; widely different cluster counts at
different δ would suggest the δ choice matters more than expected.

In [ ]:
sens_df = compute_robustness_sensitivity("LRS70D", campaign="2022_02")
print("Cluster counts per δ:")
print(sens_df.groupby("delta_m")["cluster_id"].count())

In [ ]:
fig = plot_delta_sensitivity(
    sens_df, well_id="LRS70D",
    output_path="results/sec_robustness/2022_02/figures/sensitivity_LRS70D.png",
)
plt.close(fig)
Image(filename="results/sec_robustness/2022_02/figures/sensitivity_LRS70D.png")

## 4. BIC curves across all wells

In [ ]:
fig = plot_bic_curves(
    list(WELLS.keys()),
    output_path="results/sec_robustness/2022_02/figures/bic_curves.png",
)
plt.close(fig)
Image(filename="results/sec_robustness/2022_02/figures/bic_curves.png")

## 5. Batch run for all wells

This is what ``scripts/sec_robustness_analysis.py`` runs. The CSVs are
the primary outputs you'll cite in the thesis.

In [ ]:
# Equivalent of the CLI:
# uv run python scripts/sec_robustness_analysis.py
import subprocess
result = subprocess.run(
    ["python", "scripts/sec_robustness_analysis.py"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])

In [ ]:
# Browse the per-cluster CSV after the batch
clusters_all = pd.read_csv("results/sec_robustness/2022_02/robustness_clusters.csv")
print(f"Total clusters across all wells: {len(clusters_all)}")
print()
print("Top robust BPs per well (agreement >= 5):")
top_robust = clusters_all[clusters_all["agreement"] >= 5].copy()
print(top_robust[["well_id", "depth_median", "agreement", "persistence",
                   "wide_flag", "depth_iqr"]]
      .sort_values(["well_id", "agreement"], ascending=[True, False])
      .to_string(index=False))